# Silver player master

## Pipeline overview

**Purpose:** Preserve the nflverse player-master snapshot as a clean, reusable Silver table at one row per source \`gsis_id\`. This notebook does not merge identities, infer historical teams or positions, or build player-week facts. It runs alongside the existing Silver notebooks.

| Role | File | Current columns | Grain |
| --- | --- | ---: | --- |
| Input | \`data/bronze/players/players.parquet\` | 39 | Source \`gsis_id\` |
| Output | \`data/silver/players/silver_players.parquet\` | 39 | Source \`gsis_id\` |

The input is one unpartitioned snapshot, currently 24,853 rows. Source field names and order remain unchanged. \`latest_team\`, \`status\`, \`position\`, and \`years_of_experience\` describe this snapshot, not the player's state in every historical season.

### Cleaning and transformations

1. Count and remove only exact full-row duplicates. Conflicting rows for one \`gsis_id\` stop the build.
2. Keep identifiers and \`jersey_number\` as text. Do not require \`gsis_id\` to follow one modern text pattern.
3. Parse \`birth_date\` as a nullable datetime; convert verified whole-number measurements, season/experience counts, and draft fields to nullable integers.
4. Preserve source nulls and anomalies rather than filling, merging, or correcting them without evidence.

### Validation and quality checks

- Required file and 39 named columns; non-null, nonblank, unique \`gsis_id\`; exact-duplicate and row-count reconciliation.
- Safe numeric/date conversion; \`rookie_season <= last_season\`, nonnegative experience, and draft year no later than last season.
- Source-value reconciliation outside approved type changes; same 39 columns and equal Parquet round trip.
- Non-blocking diagnostics for shared alternate IDs, nonstandard \`gsis_id\` formats, incomplete draft fields, and suspect height/weight values.

## 1. Set paths and type contract

The numeric fields below are currently whole-valued in Bronze. We convert them only after validating that no values would be rounded or lost.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


EXPECTED_COLUMNS = 39
PLAYER_ID_COLUMNS = [
    "gsis_id", "esb_id", "nfl_id", "pfr_id", "pff_id",
    "otc_id", "espn_id", "smart_id",
]
INTEGER_COLUMNS = [
    "height", "weight", "rookie_season", "last_season",
    "years_of_experience", "draft_year", "draft_round", "draft_pick",
]

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "ROADMAP.md").exists():
    PROJECT_ROOT = next(
        parent for parent in PROJECT_ROOT.parents
        if (parent / "ROADMAP.md").exists()
    )

INPUT_PATH = PROJECT_ROOT / "data" / "bronze" / "players" / "players.parquet"
OUTPUT_PATH = PROJECT_ROOT / "data" / "silver" / "players" / "silver_players.parquet"

print(f"Input: {INPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Output: {OUTPUT_PATH.relative_to(PROJECT_ROOT)}")

Input: data/bronze/players/players.parquet
Output: data/silver/players/silver_players.parquet


## 2. Inspect the source contract

This is one global player-master snapshot rather than four season partitions. Column drift is a stop condition; row count can change on refresh.

In [2]:
assert INPUT_PATH.exists(), f"Missing player master: {INPUT_PATH}"

source_schema = pq.read_schema(INPUT_PATH)
source_columns = source_schema.names
assert len(source_columns) == EXPECTED_COLUMNS
assert set(PLAYER_ID_COLUMNS + INTEGER_COLUMNS + [
    "birth_date", "display_name", "jersey_number",
    "position", "latest_team", "status",
]).issubset(source_columns)

bronze_players = pd.read_parquet(INPUT_PATH)
assert bronze_players.columns.tolist() == source_columns

pd.DataFrame([{
    "source": str(INPUT_PATH.relative_to(PROJECT_ROOT)),
    "rows": len(bronze_players),
    "columns": len(bronze_players.columns),
    "null_gsis_id": int(bronze_players["gsis_id"].isna().sum()),
    "duplicate_gsis_id": int(bronze_players.duplicated("gsis_id").sum()),
}])

,source,rows,columns,null_gsis_id,duplicate_gsis_id
0,data/bronze/players/players.parquet,24853,39,0,0


## 3. Remove exact duplicates and validate the source key

A shared name or alternate provider ID is not enough to merge records. Only the source \`gsis_id\` defines this table's grain.

In [3]:
exact_duplicate_mask = bronze_players.duplicated(keep="first")
exact_duplicate_count = int(exact_duplicate_mask.sum())
source_without_duplicates = bronze_players.loc[
    ~exact_duplicate_mask
].reset_index(drop=True)

assert len(source_without_duplicates) == len(bronze_players) - exact_duplicate_count
assert source_without_duplicates["gsis_id"].notna().all()
assert source_without_duplicates["gsis_id"].map(type).eq(str).all()
assert source_without_duplicates["gsis_id"].str.strip().ne("").all()
assert not source_without_duplicates.duplicated("gsis_id").any(), (
    "Conflicting records share a gsis_id"
)

pd.DataFrame([{
    "bronze_rows": len(bronze_players),
    "exact_duplicates_removed": exact_duplicate_count,
    "silver_rows_expected": len(source_without_duplicates),
    "unique_gsis_id": source_without_duplicates["gsis_id"].nunique(),
}])

,bronze_rows,exact_duplicates_removed,silver_rows_expected,unique_gsis_id
0,24853,0,24853,24853


## 4. Normalize dates and whole-number fields

The output retains all 39 source fields. Blank optional identifiers, undrafted-player fields, and missing birth dates remain null. \`jersey_number\` remains text.

In [4]:
silver_players = source_without_duplicates.copy()

for column in INTEGER_COLUMNS:
    numeric_values = pd.to_numeric(silver_players[column], errors="raise")
    populated = numeric_values.dropna()
    assert np.isfinite(populated).all(), (column, "non-finite")
    assert populated.mod(1).eq(0).all(), (column, "fractional")
    assert populated.abs().le(2**53).all(), (column, "unsafe float integer")
    silver_players[column] = numeric_values.astype("Int64")

silver_players["birth_date"] = pd.to_datetime(
    silver_players["birth_date"], format="%Y-%m-%d", errors="raise"
)

for column in PLAYER_ID_COLUMNS:
    populated_ids = silver_players[column].dropna()
    assert populated_ids.map(type).eq(str).all(), column

assert silver_players["jersey_number"].dropna().map(type).eq(str).all()
assert silver_players.columns.tolist() == source_columns
assert len(silver_players) == len(source_without_duplicates)

changed_columns = set(INTEGER_COLUMNS + ["birth_date"])
untouched_columns = [name for name in source_columns if name not in changed_columns]
pd.testing.assert_frame_equal(
    source_without_duplicates[untouched_columns],
    silver_players[untouched_columns],
)
for column in INTEGER_COLUMNS:
    pd.testing.assert_series_equal(
        pd.to_numeric(source_without_duplicates[column]).astype("Float64"),
        silver_players[column].astype("Float64"),
        check_names=False,
    )
assert silver_players["birth_date"].dt.strftime("%Y-%m-%d").equals(
    source_without_duplicates["birth_date"]
)

print("Selected types normalized; source values and nulls reconciled.")

Selected types normalized; source values and nulls reconciled.


## 5. Validate player-master relationships

These checks concern the source record itself. This notebook does not assert that every roster or weekly-stat player has a match in the master; that belongs to cross-source identity validation.

In [5]:
assert silver_players["gsis_id"].notna().all()
assert not silver_players.duplicated("gsis_id").any()
assert silver_players["display_name"].notna().all()
assert silver_players["position"].notna().all()
assert silver_players["rookie_season"].le(
    silver_players["last_season"]
).all()
assert silver_players["years_of_experience"].ge(0).all()

drafted = silver_players["draft_year"].notna()
assert silver_players.loc[drafted, "draft_year"].le(
    silver_players.loc[drafted, "last_season"]
).all()

print("Player grain and source chronology checks passed.")

Player grain and source chronology checks passed.


## 6. Review source exceptions without rewriting identity

Many historical source \`gsis_id\` values do not use the modern \`00-...\` format. A shared \`esb_id\` or \`smart_id\` is also not sufficient evidence to collapse rows. Physical-measurement thresholds below are review flags, not repairs or eligibility filters.

In [6]:
alternate_id_summary = []
for column in PLAYER_ID_COLUMNS[1:]:
    populated = silver_players.loc[silver_players[column].notna(), column]
    alternate_id_summary.append({
        "identifier": column,
        "populated_rows": len(populated),
        "duplicate_values": int(populated.duplicated().sum()),
    })

nonstandard_gsis = ~silver_players["gsis_id"].str.fullmatch(
    r"00-[0-9]{7}", na=False
)
suspect_measurements = (
    silver_players["height"].notna()
    & ~silver_players["height"].between(60, 85)
) | (
    silver_players["weight"].notna()
    & ~silver_players["weight"].between(100, 500)
)
draft_fields = ["draft_year", "draft_round", "draft_pick", "draft_team"]
draft_field_count = silver_players[draft_fields].notna().sum(axis=1)
incomplete_draft_rows = draft_field_count.between(1, 3)

diagnostic_summary = pd.DataFrame([{
    "nonstandard_gsis_ids": int(nonstandard_gsis.sum()),
    "suspect_measurements": int(suspect_measurements.sum()),
    "incomplete_draft_rows": int(incomplete_draft_rows.sum()),
}])
display(pd.DataFrame(alternate_id_summary))
display(diagnostic_summary)
display(
    silver_players.loc[
        silver_players["esb_id"].duplicated(keep=False)
        | silver_players["smart_id"].duplicated(keep=False),
        ["gsis_id", "display_name", "esb_id", "smart_id", "rookie_season", "last_season"],
    ].head(20)
)
display(
    silver_players.loc[
        suspect_measurements,
        ["gsis_id", "display_name", "height", "weight", "position"],
    ].head(20)
)

,identifier,populated_rows,duplicate_values
0,esb_id,24853,1
1,nfl_id,11923,0
2,pfr_id,22688,0
3,pff_id,11637,0
4,otc_id,9818,0
5,espn_id,16586,0
6,smart_id,24853,1


,nonstandard_gsis_ids,suspect_measurements,incomplete_draft_rows
0,6079,1,0


,gsis_id,display_name,esb_id,smart_id,rookie_season,last_season
6334,00-0040793,Paschal Ekeji,EKE080143,3200454b-4508-0143-f843-6dec13481f10,2026,2026
6335,EKE080143,Paschal Ekeji,EKE080143,3200454b-4508-0143-f843-6dec13481f10,2025,2025


,gsis_id,display_name,height,weight,position
18099,RAY747522,Gerry Raymond,<NA>,1,G


## 7. Persist and verify the Silver player master

A fresh Parquet read must reproduce all values, source column order, and selected normalized types.

In [7]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
silver_players.to_parquet(OUTPUT_PATH, index=False)
reloaded = pd.read_parquet(OUTPUT_PATH)

pd.testing.assert_frame_equal(reloaded, silver_players)
assert reloaded.columns.tolist() == source_columns
assert len(reloaded.columns) == EXPECTED_COLUMNS
assert reloaded["gsis_id"].notna().all()
assert not reloaded.duplicated("gsis_id").any()
assert pd.api.types.is_datetime64_any_dtype(reloaded["birth_date"])
assert all(str(reloaded[column].dtype) == "Int64" for column in INTEGER_COLUMNS)

pd.DataFrame([{
    "output": str(OUTPUT_PATH.relative_to(PROJECT_ROOT)),
    "bronze_rows": len(bronze_players),
    "exact_duplicates_removed": exact_duplicate_count,
    "silver_rows": len(reloaded),
    "columns": len(reloaded.columns),
    "unique_gsis_id": reloaded["gsis_id"].nunique(),
}])

,output,bronze_rows,exact_duplicates_removed,silver_rows,columns,unique_gsis_id
0,data/silver/players/silver_players.parquet,24853,0,24853,39,24853
